# Wersja regresji logistyczenj bez pipeline
Ten sam model tylko w innej postaci jest w regresja_logistyczna_pipeline tutaj jest przedstawione jak doszedlem do wyniku \
Wynik w tej chwili wychodzi AUC 0.75 trzeba sie zastwnowic czy to nas stysfakcjonuje.

In [79]:
#importuje napisane przez nas funkcje i je aktualizuje na bieżąco, dzięki czemu nie trzeba restartować kernela przy każdej zmianie w funkcjach
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from funkcje.funkcje import *
from funkcje.wskazniki_kredytowe import oblicz_wskazniki

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [80]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
from optbinning import OptimalBinning
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import balanced_accuracy_score 
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV

In [81]:
#Dane
df = pd.read_csv("../zbior_1.csv")
#usuwam wybrane zmienne kategoryczne na podstawie tego co w eksploracja_danych.ipynb
X=df.iloc[:, :-1].drop(columns=["szczegolnaFormaPrawna_Symbol", "schemat_wsk_rzis","schemat_wsk_bilans"])
X=oblicz_wskazniki(X)
y=df.iloc[:, -1]

#Podzil na zbior treningowy (70%) i pozostale
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=2137,
    stratify=y
)

#Podzil na zbior walidacyjny (15%) i testowy (15%)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=2137,
    stratify=y_temp
)

In [82]:
#Podzial na cechy numeryczne i kategoryczne
numeric_columns = X_train.select_dtypes(include=['float']).columns
numeric_columns = numeric_columns.append(pd.Index(['wsk_liczba_dni_istnienia']))
categorical_columns = X_train.select_dtypes(include=['str', 'int']).columns
categorical_columns = categorical_columns.drop('wsk_liczba_dni_istnienia')

X_cat=X_train[categorical_columns]
X_num=X_train[numeric_columns]

#zastopinie nieskonczonosci przez Nan
X_num = X_num.replace([np.inf, -np.inf], np.nan)

In [83]:
#dodaanie kolumny wskaznikow do X_num
X_num=oblicz_wskazniki(X_num)

In [84]:
#usuniecie kolumn o korelacji powyzej 0.9
X_num =usuwanie_najbardziej_scorrelowanych(X_num, prog=0.7, czy_wyswietlic=True)

Usunięto 141 kolumn.


In [85]:
#uniecie kolumn gdzie ponad polowa zmiennych to 0 lub Nan
maska_nan = X_num.isna().mean() <= 0.5
maska_zer = X_num.eq(0).mean() <= 0.5
maska_jakosci = maska_nan & maska_zer
X_num = X_num.loc[:, maska_jakosci]
print(f"Usunięto {len(maska_jakosci) - maska_jakosci.sum()} kolumny o niskiej jakości danych.")

Usunięto 40 kolumny o niskiej jakości danych.


In [86]:
#znalezeinie zmiennych nie monotonicznych
nie_monotoniczne = wykryj_niemonotoniczne(X_num, y_train, prog=0.1)
print(f"Znaleziono {len(nie_monotoniczne[0])} zmiennych nie monotonicznych: {nie_monotoniczne[0]}")
X_num=dodaj_cechy_zawiasowe(X_num, nie_monotoniczne)
X_valid=dodaj_cechy_zawiasowe(X_valid, nie_monotoniczne)

Znaleziono 9 zmiennych nie monotonicznych: ['Aktywa', 'wsk_mnoznik_kap_wl', 'wsk_ebitda_aktywa_3', 'wsk_pokrycie_aktywow_tr_kapitalem_st', 'wsk_udzial_kap_wlasnego_aktywa_2', 'wsk_ebitda_zobowiazan_odsetki_4', 'wsk_poziom_kosztow_operacyjnych', 'wsk_cykl_konwersji_gotowki', 'wsk_udzial_zob_biez_sprzedaz_2']


<string>:40: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
<string>:41: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
<string>:40: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
<string>:41: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times,

In [87]:
#policzenie jednowymiarowego AUC dla każdej kolumny numerycznej, Małysz twierdził, że to pomaga zobaczyć jak cech jest ważna
cechy_num = [c for c in X_num.columns]
rows = []
for c in cechy_num:

    m = X_num[c].notna()
    auc = roc_auc_score(y_train.loc[m], X_num.loc[m, c])
    rows.append({"cecha": c, "AUC jednowymiarowe": max(auc, 1 - auc), "kierunek": "+" if auc >= 0.5 else "-", "braki": 1 - m.mean()})
tab_auc = pd.DataFrame(rows).set_index("cecha").sort_values("AUC jednowymiarowe", ascending=False)
with pd.option_context("display.max_rows", None):
    display(tab_auc.round(3))
podejrzane = tab_auc.index[tab_auc["AUC jednowymiarowe"] > 0.85].tolist()
print("Podejrzanie silne cechy (AUC > 0,85):", podejrzane)

,AUC jednowymiarowe,kierunek,braki
cecha,,,
wsk_index_skr_zaw,0.700,+,0.000
wsk_index_nowe,0.694,+,0.000
wsk_ebitda_zobowiazan_2,0.656,-,0.000
wsk_marza_ebitda_1,0.654,-,0.000
wsk_marza_ebitda_2,0.654,-,0.000
wsk_rent_kaptial_podstawowy,0.653,-,0.000
wsk_akt_generowania_got_1,0.651,-,0.000
wsk_pokrycie_zob_kr_gotowkowe_1,0.651,-,0.000
wsk_pokrycia_odsetek_zyskiem,0.650,-,0.000


Podejrzanie silne cechy (AUC > 0,85): []


In [88]:
#usuniecie kolumn o AUC niższym niż 0.55 i pomnozenie wierszy z "+" przez -1 i zrobinie tego w zbiorze walidacyjnym
kolumny_do_usuniecia = tab_auc[tab_auc["AUC jednowymiarowe"] < 0.55].index.tolist()
mask_odwroc = (tab_auc["kierunek"] == "+") & (tab_auc["AUC jednowymiarowe"] >= 0.55)
kolumny_do_odwrocenia = tab_auc[mask_odwroc].index.tolist()

X_num = X_num.drop(columns=kolumny_do_usuniecia)
X_num[kolumny_do_odwrocenia] = -X_num[kolumny_do_odwrocenia]

X_valid = X_valid.drop(columns=kolumny_do_usuniecia)
X_valid[kolumny_do_odwrocenia] = -X_valid[kolumny_do_odwrocenia]

In [89]:
#policzenie jednowymiarowego AUC dla każdej kolumny numerycznej jeszcze raz
cechy_num = [c for c in X_num.columns]
rows = []
for c in cechy_num:

    m = X_num[c].notna()
    auc = roc_auc_score(y_train.loc[m], X_num.loc[m, c])
    rows.append({"cecha": c, "AUC jednowymiarowe": max(auc, 1 - auc), "kierunek": "+" if auc >= 0.5 else "-", "braki": 1 - m.mean()})
tab_auc = pd.DataFrame(rows).set_index("cecha").sort_values("AUC jednowymiarowe", ascending=False)
with pd.option_context("display.max_rows", None):
    display(tab_auc.round(3))
podejrzane = tab_auc.index[tab_auc["AUC jednowymiarowe"] > 0.85].tolist()
print("Podejrzanie silne cechy (AUC > 0,85):", podejrzane)

,AUC jednowymiarowe,kierunek,braki
cecha,,,
wsk_index_skr_zaw,0.700,-,0.000
wsk_index_nowe,0.694,-,0.000
wsk_ebitda_zobowiazan_2,0.656,-,0.000
wsk_marza_ebitda_1,0.654,-,0.000
wsk_marza_ebitda_2,0.654,-,0.000
wsk_rent_kaptial_podstawowy,0.653,-,0.000
wsk_akt_generowania_got_1,0.651,-,0.000
wsk_pokrycie_zob_kr_gotowkowe_1,0.651,-,0.000
wsk_pokrycia_odsetek_zyskiem,0.650,-,0.000


Podejrzanie silne cechy (AUC > 0,85): []


In [90]:
#zmiana zmiennych kategorycznych na numeryczne przy pomocy WoE z optbinning
for col in X_cat.columns:
    optb = OptimalBinning(
        name=col,
        dtype="categorical",
        solver="cp",
        cat_cutoff=0.01,
        max_n_bins=4
    )

    optb.fit(X_cat[col], y_train)

    X_cat[col] = optb.transform(
        X_cat[col], metric="woe"
    )
    X_valid[col] = optb.transform(
        X_valid[col], metric="woe"
    )



In [91]:
#policzenie jednowymiarowego AUC dla każdej kolumny kategorycznej
cechy_num = [c for c in X_cat.columns]
rows = []
for c in cechy_num:

    m = X_cat[c].notna()
    auc = roc_auc_score(y_train.loc[m], X_cat.loc[m, c])
    rows.append({"cecha": c, "AUC jednowymiarowe": max(auc, 1 - auc), "kierunek": "+" if auc >= 0.5 else "-", "braki": 1 - m.mean()})
tab_auc = pd.DataFrame(rows).set_index("cecha").sort_values("AUC jednowymiarowe", ascending=False)
with pd.option_context("display.max_rows", None):
    display(tab_auc.round(3))
podejrzane = tab_auc.index[tab_auc["AUC jednowymiarowe"] > 0.85].tolist()
print("Podejrzanie silne cechy (AUC > 0,85):", podejrzane)

,AUC jednowymiarowe,kierunek,braki
cecha,,,
pkdKod,0.575,-,0.0
formaWlasnosci_Symbol,0.535,-,0.0


Podejrzanie silne cechy (AUC > 0,85): []


In [92]:
#znalezienie kolumn, do przeskalowania
skosne_cechy = X_num.skew().sort_values(ascending=False)
do_przeskalowania = skosne_cechy[(skosne_cechy > 2) & (X_num.min() >= 0)]
print("Cechy do przeskalowania:", do_przeskalowania.index.tolist())

Cechy do przeskalowania: ['Aktywa_wzrost', 'wsk_pokrycie_aktywow_tr_kapitalem_st_spadek', 'wsk_kredyt_plynnosc_biezaca', 'wsk_ebitda_zobowiazan_odsetki_4_wzrost']


In [93]:
#zlogrytmowanie cech o skośności większej niż 2
for col in do_przeskalowania.index.tolist():
    X_num[col] = np.log1p(X_num[col])
    X_valid[col] = X_valid[col].clip(lower=0)
    X_valid[col] = np.log1p(X_valid[col])
#polaczenie cech numerycznych i kategorycznych w jeden zbiór
#X_train_final = pd.concat([X_num, X_cat], axis=1)
#zostawiam tylko numeryczne bo kategoryczne sa slabe
X_train_final=X_num

In [94]:
#napisanie pipeline, który będzie przeskalowywał cechy i inputował nany i trenował regresję logistyczną z regularizacją l1
pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(l1_ratio=1, solver='liblinear',C=1, random_state=2137))
])

In [95]:
X_valid = X_valid[X_train_final.columns]
X_valid = X_valid.replace([np.inf, -np.inf], np.nan)

In [96]:
pipeline.fit(X_train_final, y_train)
pred=pipeline.predict_proba(X_valid)[:, 1]

In [97]:
#policzenie AUC na zbiorze walidacyjnym
roc_auc_score(y_valid, pred)

0.8089024390243903